1. Replacing Data — `CREATE OR REPLACE TABLE` vs `INSERT OVERWRITE`
2. Appending Data — `INSERT INTO`
3. Merging Data — `MERGE INTO` (upsert, insert-only merge)

## Setup

In [0]:
catalog = "main"
schema  = "school"
volume  = "raw_data"

# Path base (Volumes)
base_path = f"/Volumes/{catalog}/{schema}/{volume}"

# Paths
students_path  = f"{base_path}/students-json"
courses_path   = f"{base_path}/courses-csv"

# New Paths
students_new_path    = f"{base_path}/students-json-new"
enrollments_path     = f"{base_path}/enrollments-parquet"
enrollments_new_path = f"{base_path}/enrollments-parquet-new"
courses_new_path     = f"{base_path}/courses-csv-new"

print("Base path:", base_path)
print("Students path:", students_path)
print("Courses path:",  courses_path)

Base path: /Volumes/main/school/raw_data
Students path: /Volumes/main/school/raw_data/students-json
Courses path: /Volumes/main/school/raw_data/courses-csv


In [0]:
# Create structure if it does not exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

print("List structure")

List structure


In [0]:
# S3 Source
s3_base = "s3://dalhussein-books/DEA-Book/datasets/school/v1"

s3_students = f"{s3_base}/students-json"
s3_courses = f"{s3_base}/courses-csv"
s3_students_new = f"{s3_base}/students-json-new"
s3_enrollments = f"{s3_base}/enrollments"
s3_enrollments_new = f"{s3_base}/enrollments-new"
s3_courses_new = f"{s3_base}/courses-csv-new"

# Copy to Volumes
dbutils.fs.cp(s3_students, students_path, recurse=True)
dbutils.fs.cp(s3_courses, courses_path, recurse=True)
dbutils.fs.cp(s3_students_new, students_new_path, recurse=True)
dbutils.fs.cp(s3_enrollments, enrollments_path, recurse=True)
dbutils.fs.cp(s3_enrollments_new, enrollments_new_path, recurse=True)
dbutils.fs.cp(s3_courses_new, courses_new_path, recurse=True)

print("Data copied to Volumes")

Data copied to Volumes


In [0]:
# Check files
print("=== Students (JSON) ===")
display(dbutils.fs.ls(students_path))

print("\n=== Courses (CSV) ===")
display(dbutils.fs.ls(courses_path))

=== Students (JSON) ===


path,name,size,modificationTime
dbfs:/Volumes/main/school/raw_data/students-json/export_001.json,export_001.json,82932,1777337808000
dbfs:/Volumes/main/school/raw_data/students-json/export_002.json,export_002.json,83399,1777337809000
dbfs:/Volumes/main/school/raw_data/students-json/export_003.json,export_003.json,83232,1777337809000
dbfs:/Volumes/main/school/raw_data/students-json/export_004.json,export_004.json,83489,1777337810000
dbfs:/Volumes/main/school/raw_data/students-json/export_005.json,export_005.json,83208,1777337810000
dbfs:/Volumes/main/school/raw_data/students-json/export_006.json,export_006.json,55562,1777337811000



=== Courses (CSV) ===


path,name,size,modificationTime
dbfs:/Volumes/main/school/raw_data/courses-csv/export_001.csv,export_001.csv,202,1777337815000
dbfs:/Volumes/main/school/raw_data/courses-csv/export_002.csv,export_002.csv,211,1777337815000
dbfs:/Volumes/main/school/raw_data/courses-csv/export_003.csv,export_003.csv,203,1777337816000
dbfs:/Volumes/main/school/raw_data/courses-csv/export_004.csv,export_004.csv,206,1777337816000


---
## Base Table (`enrollments`)

We need the `enrollments` table as a starting point.
We create it using CTAS from the Parquet (We use it with students).

In [0]:
%sql
-- Starting point: we create enrollments from Parquet
CREATE TABLE IF NOT EXISTS main.school.enrollments
AS SELECT * 
FROM parquet.`/Volumes/main/school/raw_data/enrollments-parquet`

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS total 
FROM main.school.enrollments

total
3550


---
## 1. Replacing Data

Databricks offers **two ways** to completely replace the contents of a Delta table:

| Method | Creates table if it doesn't exist | Changes schema | Preserves Delta history |
|---|---|---|---|
`CREATE OR REPLACE TABLE` | ✅ Yes | ✅ Yes | ✅ Yes (new version) |
`INSERT OVERWRITE` | ❌ No | ❌ No — schema enforcement | ✅ Yes (new version) |

> 💡 **Both are superior to DROP + CREATE** because they preserve Delta history
> and guarantee ACID atomicity.

### 1a. `CREATE OR REPLACE TABLE` (CRAS)

In [0]:
%sql
-- State BEFORE replacement
SELECT COUNT(*) AS total_before 
FROM main.school.enrollments

total_before
3550


In [0]:
%sql
-- CRAS: Replaces data and schema in an atomic operation
-- If the table does not exist → creates it. If it exists → generates a new version in the transaction log.

CREATE OR REPLACE TABLE main.school.enrollments
AS SELECT * 
FROM parquet.`/Volumes/main/school/raw_data/enrollments-parquet`

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- The history shows version 0 (original CTAS) and version 1 (REPLACE)
-- Key difference vs DROP + CREATE: the history does NOT disappear
DESCRIBE HISTORY main.school.enrollments

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-04-28T00:57:44.000Z,71752003934143,davidcaleb1998@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3033934490663247),9e798d68-e629-4ba6-aa5d-c8cc4024cccb,0428-005607-fnkutb7y-v2n,4,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 3, numRemovedBytes -> 52553, numDeletionVectorsRemoved -> 0, numOutputRows -> 2150, numOutputBytes -> 32382)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
4,2026-04-27T23:18:03.000Z,71752003934143,davidcaleb1998@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3033934490663247),fb78c928-9baa-4902-9274-f48ce83329be,0427-222028-7p2tzeim-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 700, numOutputBytes -> 9996)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
3,2026-04-27T23:17:59.000Z,71752003934143,davidcaleb1998@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3033934490663247),698d7bac-5be2-4580-8b9e-2fa54ac70475,0427-222028-7p2tzeim-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 700, numOutputBytes -> 9996)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
2,2026-04-27T23:17:53.000Z,71752003934143,davidcaleb1998@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> true, partitionBy -> [])",null,List(3033934490663247),b7e38e5b-7595-4134-9ebf-1fdab1eb7da9,0427-222028-7p2tzeim-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 32382, numDeletionVectorsRemoved -> 0, numOutputRows -> 2150, numOutputBytes -> 32561)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-04-27T23:17:48.000Z,71752003934143,davidcaleb1998@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3033934490663247),19b4a764-70ec-4050-a66d-277ae9779129,0427-222028-7p2tzeim-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 32382, numDeletionVectorsRemoved -> 0, numOutputRows -> 2150, numOutputBytes -> 32382)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-04-27T23:17:43.000Z,71752003934143,davidcaleb1998@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3033934490663247),111777df-7f99-4fb2-b4c7-609fe415551d,0427-222028-7p2tzeim-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2150, numOutputBytes -> 32382)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


> 📌 **Time Travel is still available:**
> ```sql
> SELECT * 
> FROM main.school.enrollments 
> VERSION AS OF 0
> ```
> With DROP + CREATE this would be impossible, the history would be deleted.

### 1b. `INSERT OVERWRITE`

In [0]:
%sql
-- INSERT OVERWRITE: replaces the data but respects the existing schema
-- Delta registers this operation as WRITE with mode = Overwrite
INSERT OVERWRITE main.school.enrollments
SELECT * 
FROM parquet.`/Volumes/main/school/raw_data/enrollments-parquet`

num_affected_rows,num_inserted_rows
2150,2150


In [0]:
%sql
-- In the history it appears as operation = WRITE, mode = Overwrite
-- Difference vs CRAS: appears as CREATE TABLE vs WRITE
DESCRIBE HISTORY main.school.enrollments

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-04-28T00:57:54.000Z,71752003934143,davidcaleb1998@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> true, partitionBy -> [])",null,List(3033934490663247),dca94569-e7b3-4ec9-8b59-aabc7afccbb2,0428-005607-fnkutb7y-v2n,5,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 32382, numDeletionVectorsRemoved -> 0, numOutputRows -> 2150, numOutputBytes -> 32561)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
5,2026-04-28T00:57:44.000Z,71752003934143,davidcaleb1998@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3033934490663247),9e798d68-e629-4ba6-aa5d-c8cc4024cccb,0428-005607-fnkutb7y-v2n,4,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 3, numRemovedBytes -> 52553, numDeletionVectorsRemoved -> 0, numOutputRows -> 2150, numOutputBytes -> 32382)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
4,2026-04-27T23:18:03.000Z,71752003934143,davidcaleb1998@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3033934490663247),fb78c928-9baa-4902-9274-f48ce83329be,0427-222028-7p2tzeim-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 700, numOutputBytes -> 9996)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
3,2026-04-27T23:17:59.000Z,71752003934143,davidcaleb1998@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3033934490663247),698d7bac-5be2-4580-8b9e-2fa54ac70475,0427-222028-7p2tzeim-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 700, numOutputBytes -> 9996)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
2,2026-04-27T23:17:53.000Z,71752003934143,davidcaleb1998@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> true, partitionBy -> [])",null,List(3033934490663247),b7e38e5b-7595-4134-9ebf-1fdab1eb7da9,0427-222028-7p2tzeim-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 32382, numDeletionVectorsRemoved -> 0, numOutputRows -> 2150, numOutputBytes -> 32561)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-04-27T23:17:48.000Z,71752003934143,davidcaleb1998@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3033934490663247),19b4a764-70ec-4050-a66d-277ae9779129,0427-222028-7p2tzeim-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 32382, numDeletionVectorsRemoved -> 0, numOutputRows -> 2150, numOutputBytes -> 32382)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-04-27T23:17:43.000Z,71752003934143,davidcaleb1998@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3033934490663247),111777df-7f99-4fb2-b4c7-609fe415551d,0427-222028-7p2tzeim-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2150, numOutputBytes -> 32382)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


---
## 2. Appending Data (`INSERT INTO`)

`INSERT INTO` adds new rows without affecting existing ones.

It is the simplest append method, but **it does not have duplicate protection**.

In [0]:
%sql
SELECT COUNT(*) AS total_before 
FROM main.school.enrollments

total_before
2150


In [0]:
%sql
-- We add the new records to the end of the table
INSERT INTO main.school.enrollments
SELECT * 
FROM parquet.`/Volumes/main/school/raw_data/enrollments-parquet-new`

num_affected_rows,num_inserted_rows
700,700


In [0]:
%sql
SELECT COUNT(*) AS total_after 
FROM main.school.enrollments

total_after
2850


In [0]:
%sql
-- We execute INSERT INTO a SECOND time to demonstrate the problem
INSERT INTO main.school.enrollments
SELECT * 
FROM parquet.`/Volumes/main/school/raw_data/enrollments-parquet-new`

num_affected_rows,num_inserted_rows
700,700


In [0]:
%sql
-- The total increased again → there are DUPLICATES
SELECT COUNT(*) AS total_with_duplicates 
FROM main.school.enrollments

total_with_duplicates
3550


> ⚠️ **Limitation of `INSERT INTO`:**
> If the pipeline fails and retries, or executes twice, the records are duplicated.

> **Solution → `MERGE INTO`** (next section)

---
## 3. Merging Data — `MERGE INTO`

`MERGE INTO` executes **INSERT + UPDATE + DELETE** in a single atomic operation. It compares the source against the destination by a key and decides what to do with each row:

```
WHEN MATCHED → UPDATE or DELETE (exists in both)
WHEN NOT MATCHED → INSERT (new in source, not in target)
WHEN NOT MATCHED BY SOURCE → UPDATE or DELETE (exists in target but not in source)
```

### 3a. MERGE with UPDATE + INSERT — table `students`

**Scenario:** We receive corrected emails and new students.

We update only the null emails and insert the new ones — without duplication.

In [0]:
%sql
CREATE OR REPLACE TABLE main.school.students AS 
SELECT *
FROM json.`/Volumes/main/school/raw_data/students-json/*`

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * 
FROM main.school.students

email,gpa,profile,student_id,updated
dabby2y@japanpost.jp,1.48,"{""first_name"":""Dniren"",""last_name"":""Abby"",""gender"":""Female"",""address"":{""street"":""768 Mesta Terrace"",""city"":""Annecy"",""country"":""France""}}",S00001,2021-12-14T23:15:43.375Z
eabbysc1@github.com,3.02,"{""first_name"":""Etti"",""last_name"":""Abbys"",""gender"":""Female"",""address"":{""street"":""1748 Vidon Plaza"",""city"":""Varge Mondar"",""country"":""Portugal""}}",S00002,2021-12-14T23:15:43.375Z
rabelovd1@wikispaces.com,3.31,"{""first_name"":""Ronnie"",""last_name"":""Abelov"",""gender"":""Male"",""address"":{""street"":""363 Randy Park"",""city"":""San Celestio"",""country"":""Philippines""}}",S00003,2021-12-14T23:15:43.375Z
rabels9g@behance.net,1.89,"{""first_name"":""Ray"",""last_name"":""Abels"",""gender"":""Female"",""address"":{""street"":""613 Lyons Way"",""city"":""Oudtshoorn"",""country"":""South Africa""}}",S00004,2021-12-14T23:15:43.375Z
sabendrothin@cargocollective.com,3.55,"{""first_name"":""Shanon"",""last_name"":""Abendroth"",""gender"":""Female"",""address"":{""street"":""30292 Manufacturers Junction"",""city"":""Ani-e"",""country"":""Philippines""}}",S00005,2021-12-14T23:15:43.375Z
null,2.9,"{""first_name"":""Norman"",""last_name"":""Abernethy"",""gender"":""Male"",""address"":{""street"":""9292 Oxford Center"",""city"":""Gibara"",""country"":""Cuba""}}",S00006,2021-12-14T23:15:43.375Z
sabrahmson3h@blinklist.com,2.96,"{""first_name"":""Skell"",""last_name"":""Abrahmson"",""gender"":""Male"",""address"":{""street"":""90941 Hallows Park"",""city"":""Huarong Chengguanzhen"",""country"":""United Kingdom""}}",S00007,2021-12-14T23:15:43.375Z
dacheson2h@mapy.cz,1.2,"{""first_name"":""Darsey"",""last_name"":""Acheson"",""gender"":""Non-binary"",""address"":{""street"":""29579 Grim Plaza"",""city"":""Dārayyā"",""country"":""Syria""}}",S00008,2021-12-14T23:15:43.375Z
fackwoodji@gravatar.com,1.96,"{""first_name"":""Fredrick"",""last_name"":""Ackwood"",""gender"":""Male"",""address"":{""street"":""67 Dunning Plaza"",""city"":""Santo Domingo"",""country"":""Cuba""}}",S00009,2021-12-14T23:15:43.375Z
null,1.39,"{""first_name"":""Doralynne"",""last_name"":""Adamkiewicz"",""gender"":""Female"",""address"":{""street"":""84126 Glendale Center"",""city"":""Ugep"",""country"":""Nigeria""}}",S00010,2021-12-14T23:15:43.375Z


In [0]:
%sql
SELECT COUNT(*) AS total_before 
FROM main.school.students

total_before
1700


In [0]:
%sql
-- Temporary view with the input data
CREATE OR REPLACE TEMP VIEW students_updates AS
SELECT * 
FROM json.`/Volumes/main/school/raw_data/students-json-new`

In [0]:
%sql
SELECT * 
FROM students_updates 
LIMIT 5

email,gpa,profile,student_id,updated
jabbypf@webeden.co.uk,1.56,"{""first_name"":""Jacquelyn"",""last_name"":""Abby"",""gender"":""Female"",""address"":{""street"":""0 Elmside Court"",""city"":""Gornji Milanovac"",""country"":""Serbia""}}",S01701,2022-09-14T23:39:06.531Z
sabrahamssonpq@princeton.edu,1.35,"{""first_name"":""Shellysheldon"",""last_name"":""Abrahamsson"",""gender"":""Male"",""address"":{""street"":""76 Reinke Avenue"",""city"":""Phra Pradaeng"",""country"":""Thailand""}}",S01702,2022-09-14T23:39:06.531Z
jabreheartrb@usatoday.com,3.77,"{""first_name"":""Joni"",""last_name"":""Abreheart"",""gender"":""Female"",""address"":{""street"":""9256 Banding Way"",""city"":""Citambal"",""country"":""Indonesia""}}",S01703,2022-09-14T23:39:06.531Z
iadanph@digg.com,2.61,"{""first_name"":""Isobel"",""last_name"":""Adan"",""gender"":""Female"",""address"":{""street"":""073 Moulton Crossing"",""city"":""Vinsady"",""country"":""Russia""}}",S01704,2022-09-14T23:39:06.531Z
null,2.8,"{""first_name"":""Jules"",""last_name"":""Agnolo"",""gender"":""Male"",""address"":{""street"":""067 Elka Center"",""city"":""Zhongzhang"",""country"":""United Kingdom""}}",S01705,2022-09-14T23:39:06.531Z


In [0]:
%sql
MERGE INTO main.school.students AS target
USING students_updates AS source
ON target.student_id = source.student_id

-- Update email only if target has NULL and source has value
WHEN MATCHED 
  AND target.email IS NULL 
  AND source.email IS NOT NULL
THEN 
  UPDATE SET 
    target.email = source.email, 
    target.updated = source.updated

-- Insert new students that do not exist in the target
WHEN NOT MATCHED THEN 
  INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
301,100,0,201


In [0]:
%sql
-- History shows numTargetRowsUpdated and numTargetRowsInserted
DESCRIBE HISTORY main.school.students

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-04-28T00:58:57.000Z,71752003934143,davidcaleb1998@gmail.com,MERGE,"Map(predicate -> [""(student_id#13133 = student_id#13111)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(isnull(email#13130) AND isnotnull(email#13108))"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3033934490663247),fc542158-c464-4bbe-b441-32012e30366f,0428-005607-fnkutb7y-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 21427, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 100, executionTimeMs -> 6779, materializeSourceTimeMs -> 451, numTargetRowsInserted -> 201, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2894, numTargetRowsUpdated -> 100, numOutputRows -> 301, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 301, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3303)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
2,2026-04-28T00:58:28.000Z,71752003934143,davidcaleb1998@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3033934490663247),3a1ba543-5ac0-4440-8c5c-97db62ed2de2,0428-005607-fnkutb7y-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 3, numRemovedBytes -> 97402, numDeletionVectorsRemoved -> 1, numOutputRows -> 1700, numOutputBytes -> 75975)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-04-27T23:18:21.000Z,71752003934143,davidcaleb1998@gmail.com,MERGE,"Map(predicate -> [""(student_id#23539 = student_id#23517)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(isnull(email#23536) AND isnotnull(email#23514))"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3033934490663247),1179735a-cef4-49fc-bdbe-dbbc28a9d223,0427-222028-7p2tzeim-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 21427, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 100, executionTimeMs -> 3110, materializeSourceTimeMs -> 218, numTargetRowsInserted -> 201, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1109, numTargetRowsUpdated -> 100, numOutputRows -> 301, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 301, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1732)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-04-27T23:18:08.000Z,71752003934143,davidcaleb1998@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3033934490663247),0ba4ac75-89ae-41e2-858c-8df8c18b7483,0427-222028-7p2tzeim-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1700, numOutputBytes -> 75975)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT COUNT(*) AS total_after 
FROM main.school.students

total_after
1901


> 📌 **MERGE is idempotent:**
> If you run MERGE a second time, the result is **0 updates and 0 inserts**
> because the conditions (`email IS NULL`, `NOT MATCHED`) are no longer met.
> Compare this to `INSERT INTO`, which would have duplicated the records.

### 3b. Insert-only Merge (`courses`) table

**Scenario:** New courses in CSV format. Insert only those that do not exist
and that are in the *Computer Science* category, without duplicates.

In [0]:
%sql
-- Temporary view from CSV
CREATE OR REPLACE TEMP VIEW courses_updates 
(
  course_id STRING, 
  title STRING, 
  instructor STRING, 
  category STRING, 
  price DOUBLE)
USING CSV
OPTIONS ( 
  path = '/Volumes/main/school/raw_data/courses-csv-new', 
  header = 'true', 
  delimiter = ';'
)

In [0]:
%sql
-- Distribution of categories in new courses
SELECT category, COUNT(*) AS total
FROM courses_updates
GROUP BY category
ORDER BY total DESC

category,total
Computer Science,3
Food,2


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW courses_tmp_vw
  (
    course_id STRING, 
    title STRING, 
    instructor STRING,
    category STRING,
    price DOUBLE
  ) USING CSV
OPTIONS (
  path = "/Volumes/main/school/raw_data/courses-csv/export_*.csv",
  header = "true",
  delimiter = ";"
)

In [0]:
%sql
SELECT * FROM courses_tmp_vw

course_id,title,instructor,category,price
C01,Data Structures and Algorithms,Tracy N.,Computer Science,49.0
C02,JavaScript Design Patterns,Ali M.,Computer Science,28.0
C03,Neural Network,Adam R.,Computer Science,35.0
C04,Robot Dynamics and Control,Mark G.,Computer Science,20.0
C05,Python Programming,Luciano C.,Computer Science,47.0
C06,Deep Learning,François R.,Computer Science,22.0
C07,Machine Learning,Andriy R.,Computer Science,33.0
C08,Quantum Computing,Chris N.,Computer Science,41.0
C09,Advanced Data Structures,Pierre B.,Computer Science,24.0
C10,Database Design Solutions,Julia S.,Computer Science,44.0


In [0]:
%sql
CREATE OR REPLACE TABLE main.school.courses
AS
SELECT * FROM courses_tmp_vw

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS total_before 
FROM main.school.courses

total_before
12


In [0]:
%sql
-- IDEMPOTENCE PROOF: second run → 0 rows inserted
MERGE INTO main.school.courses AS target
USING courses_updates AS source
ON target.course_id = source.course_id
AND target.title = source.title

WHEN NOT MATCHED 
  AND source.category = 'Computer Science'
THEN 
  INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,0,0,3


> 📌 The second execution inserted **0 rows** — the courses already exist.
> This is the behavior we want in production pipelines.

### 3c. Complete MERGE — the 3 clauses (SCD pattern)

This is the pattern we saw in the **SCD Type 1/2/3** sessions.

The 3 clauses cover all possible cases:

In [0]:
%sql
MERGE INTO main.school.students AS target
USING students_updates AS source
ON target.student_id = source.student_id

-- Case 1: exists in both and the email changed → update
WHEN MATCHED 
  AND target.email <> source.email
THEN 
  UPDATE SET 
    target.email = source.email, 
    target.updated = source.updated

-- Case 2: exists in source but not in target → insert
WHEN NOT MATCHED THEN 
INSERT *

-- Case 3: exists in target but no longer arrives in source
-- SCD Type 1 → physical DELETE
-- SCD Type 2 → soft close (valid_to = today, is_current = false)
-- WHEN NOT MATCHED BY SOURCE THEN DELETE

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


## CLEAN UP

In [0]:
def clean_up(): 
    print("Deleting tables...") 
    spark.sql("DROP TABLE IF EXISTS main.school.students") 
    spark.sql("DROP TABLE IF EXISTS main.school.courses") 
    spark.sql("DROP TABLE IF EXISTS main.school.enrollments") 
    spark.sql("DROP TABLE IF EXISTS main.school.courses_csv") 
    spark.sql("DROP TABLE IF EXISTS main.school.courses_unparsed") 

    print("Deleting volume...") 
    dbutils.fs.rm(base_path, True) 

    print("Deleting schema...") 
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE") 

    print("Done")

In [0]:
clean_up()

Deleting tables...
Deleting volume...
Deleting schema...
Done
